In [ ]:
import opt_samp as opt

import pyreadr # Reads in R data structures
import numpy as np

In [ ]:
# Tidy Data
# Each column forms a variable
# Each row forms an observation
# For grid-based geodata, each column is a layer, and each row is a pixel. Location can be included as columns as well.

In [ ]:
# Load the covariates from the original paper for testing the algorithm
# Replace covs with your own tidy data described above

result = pyreadr.read_r('data/covs.rda')
data = result["covs"]
covs = data.to_numpy()

# Experiment setup
s_reps = 4 # number of duplicate reps to run
clhs_iter = 100000 # 100000 for the paper, could be less if needed for runtime
cpus = 18 # number of cores to use for multithreading. Change as needed for your computer.
conf = 0.95 # confidence cutoff for the dispersion

# Construct the testing space. The optimal value should be somewhere in this space.
cseq = np.array([10, 25, 50, 100, 150, 200, 250, 300, 350, 400]).repeat(10) # for the paper

# R and numpy both index like var[col, row], but in np.shape, print is (#rows, #cols)
print(covs.shape)

In [ ]:
# First, run either clhs or fscs to build the analysis. This may take some time.

if __name__ == "__main__": # needed for multithreading in most cases
    plan_results = opt.clhs_setup(cpus, clhs_iter, cseq, covs)
    # plan_results = opt.fscs_setup(cpus, clhs_iter, cseq, covs)

In [ ]:
# Next, calculate the number of bins that best suits the covariates.

# bins = opt.calculate_bins(covs) # Use this instead if you want to calculate the bins from the covariates
bins = 30 # set to 30 for the paper

In [ ]:
# Calculate the different matrices needed for the divergence, and calculate the three divergence results.

quantiles = opt.calculate_quantiles(bins, covs)
cov_matrix = opt.calculate_covariates(bins, quantiles, covs)
kldiv_storage, jsdiv_storage, jsdist_storage = opt.calculate_divergences(plan_results, bins, covs, quantiles, cov_matrix)

In [ ]:
# Plot the results, including the optimal values from each metric

opt.plot_divergences(cseq, conf, kldiv_storage, jsdiv_storage, jsdist_storage)